In [10]:
from kiteconnect import KiteConnect
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service as ChromeService
import time
import os
from pyotp import TOTP

In [11]:
totp = TOTP("")
token = totp.now()
print(token)

762706


In [12]:
def autologin():
    token_path = "api_key.txt"
    key_secret = open(token_path, 'r').read().split()
    
    kite = KiteConnect(api_key=key_secret[0])
    
    # Set up Chrome options
    options = webdriver.ChromeOptions()
    # Uncomment the next line to run headless
    # options.add_argument('--headless')
    
    # Create a service object
    service = ChromeService(executable_path='./chromedriver')
    service.start()
    
    # Create a new instance of the Chrome driver
    driver = webdriver.Chrome(service=service, options=options)
    
    driver.get(kite.login_url())
    driver.implicitly_wait(10)
    
    # Enter username and password
    username = driver.find_element(By.XPATH, '//*[@id="userid"]')  # Updated XPath
    password = driver.find_element(By.XPATH, '//*[@id="password"]')  # Updated XPath
    username.send_keys(key_secret[2])
    password.send_keys(key_secret[3])
    
    # Click on the login button
    driver.find_element(By.XPATH, '//button[@type="submit"]').click()
    
    # Enter TOTP
    time.sleep(5)  # Wait for the PIN input to appear
    pin = driver.find_element(By.XPATH, '//*[@id="userid"]')  # Updated XPath
    totp = TOTP(key_secret[4])
    token = totp.now()
    pin.send_keys(token)
    
    # Click on the submit button
    driver.find_element(By.XPATH, '//button[@type="submit"]').click()
    
    # Wait for the redirect
    time.sleep(10)
    
    # Extract request token from the URL
    request_token = driver.current_url.split('request_token=')[1][:32]
    with open('request_token.txt', 'w') as the_file:
        the_file.write(request_token)
    
    driver.quit()

autologin()